<a href="https://colab.research.google.com/github/tantaiskkh/AIB-tantai/blob/main/insightface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import shutil

# ลบโฟลเดอร์ drive เก่าออก
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')

# mount ใหม่
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install insightface onnxruntime-gpu opencv-python huggingface_hub -q

from huggingface_hub import hf_hub_download
from insightface.app import FaceAnalysis
import insightface, cv2, os

# ดาวน์โหลด model
os.makedirs('/root/.insightface/models', exist_ok=True)
model_path = hf_hub_download(
    repo_id="ezioruan/inswapper_128.onnx",
    filename="inswapper_128.onnx",
    local_dir="/root/.insightface/models"
)

#path
ORIGINAL_VIDEO_FOLDER = '/content/drive/MyDrive/Data/original_video'
FACE_FOLDER           = '/content/drive/MyDrive/Data/original_face'
OUTPUT_FOLDER         = '/content/drive/MyDrive/Data/deepfake_video/insightface'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# โหลด model
app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))
swapper = insightface.model_zoo.get_model(model_path)

TARGET_FPS  = 15
MAX_SECONDS = 10

def swap_video(source_img_path, target_video_path, output_path):
    src_img   = cv2.imread(source_img_path)
    src_faces = app.get(src_img)
    if not src_faces:
        print(f"❌ ไม่เจอหน้าใน {source_img_path}")
        return False

    cap = cv2.VideoCapture(target_video_path)
    orig_fps       = cap.get(cv2.CAP_PROP_FPS) or 25
    w              = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h              = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    max_frames     = int(MAX_SECONDS * orig_fps)
    frame_interval = max(1, int(orig_fps / TARGET_FPS))

    out = cv2.VideoWriter(output_path,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          TARGET_FPS, (w, h))
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= max_frames:
            break
        if frame_idx % frame_interval == 0:
            dst_faces = app.get(frame)
            if dst_faces:
                frame = swapper.get(frame, dst_faces[0],
                                    src_faces[0], paste_back=True)
            out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    return True

# รัน 5 คลิป
orig_videos  = [f for f in os.listdir(ORIGINAL_VIDEO_FOLDER)
                if f.endswith('.mp4')][:50]
source_faces = [f for f in os.listdir(FACE_FOLDER)
                if f.endswith(('.jpg', '.jpeg', '.png'))]

print(f"📁 original_video: {len(orig_videos)} คลิป")
print(f"🧑 original_face : {len(source_faces)} รูป")

if len(orig_videos) == 0:
    print("❌ ไม่เจอวิดีโอใน original_video/")
elif len(source_faces) == 0:
    print("❌ ไม่เจอรูปหน้าใน original_face/")
else:
    for i, video_file in enumerate(orig_videos):
        src_face = source_faces[i % len(source_faces)]
        target   = os.path.join(ORIGINAL_VIDEO_FOLDER, video_file)
        output   = os.path.join(OUTPUT_FOLDER, f'deepfake_{i:03d}.mp4')

        print(f"\n🔄 [{i+1}/5] {video_file} + {src_face}")
        success = swap_video(
            os.path.join(FACE_FOLDER, src_face),
            target, output
        )
        if success:
            print(f"✅ บันทึก: deepfake_{i:03d}.mp4")

    print(f"\n🎉 เสร็จ! ไฟล์อยู่ที่ {OUTPUT_FOLDER}")
    print(os.listdir(OUTPUT_FOLDER))

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o